# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [8]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver
from time import perf_counter
from IPython.display import display,Markdown

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    forward_chain_closure,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
print("Example positive atom:", atom('Is', 3, 2, 4))
print("Example explicit exclusion:", atom('Not', 3, 2, 4))

Example positive atom: Is3_2_4
Example explicit exclusion: Not3_2_4


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
general_kb = build_general_kb(n, box_h, box_w, givens)
definite_kb = build_definite_kb(n, box_h, box_w, givens)
assert isinstance(general_kb, PropKB)
assert isinstance(definite_kb, PropDefiniteKB)
assert all(is_definite_clause(clause) for clause in definite_kb.clauses)
assert len(definite_kb.clauses) == len(set(definite_kb.clauses))
print(f"General KB: {len(general_kb.clauses):,} clauses; "
      f"{len(prop_symbols(associate('&', general_kb.clauses)))} symbols")
print(f"Definite KB: {len(definite_kb.clauses):,} facts/rules")
print("Every definite clause has the required form; no duplicate rules.")

General KB: 11,775 clauses; 729 symbols
Definite KB: 21,171 facts/rules
Every definite clause has the required form; no duplicate rules.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
start = time.time()
general_kb = build_general_kb(n, box_h, box_w, givens)
build_time = time.time() - start
print("Build time:",build_time)
print("Number of clauses:",len(general_kb.clauses))
(r,c),v = next(iter(givens.items()))
# r, c, v = 2, 2, givens.get((2, 2), 1)  # try given cell/value pair
query = atom('Is', r, c, v)
print("Query:",query)
# print(pl_resolution(general_kb, query))    
print(tt_entails(associate('&', general_kb.clauses), query)) 

Build time: 0.15380096435546875
Number of clauses: 11775
Query: Is2_1_2


KeyboardInterrupt: 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


The experiment used the default puzzle at index 0, which contains 30 given cells. Construction of the general knowledge base took approximately 0.157 seconds. It produced 11,775 stored clauses, 10,317 distinct clauses and 729 propositional symbols.

The tested query was Is1_2_3, which is one of the given facts. pl_resolution did not return an answer within the 30-second limit and was stopped after approximately 30.08 seconds. tt_entails was tested separately on the same query and was stopped after approximately 30.01 seconds. A timeout is a resource outcome rather than a logical False result.

The results show that knowledge-base construction is inexpensive compared with exhaustive inference. Since neither general inference procedure completed even for a directly given fact, neither was used to solve the complete 81-cell grid.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# A small, independent example checks the AND/OR behavior before Sudoku.
cyclic_kb = PropDefiniteKB()
for clause in ['B ==> A', 'A ==> B', 'C ==> A', '(A & B) ==> Q', 'C']:
    cyclic_kb.tell(expr(clause))
assert pl_bc_entails(cyclic_kb, expr('Q')) is True
assert pl_bc_entails(cyclic_kb, expr('Missing')) is False
unsupported_cycle = PropDefiniteKB()
for clause in ['A ==> B', 'B ==> A']:
    unsupported_cycle.tell(expr(clause))
assert pl_bc_entails(unsupported_cycle, expr('A')) is False
print('Supported cycle: True; unsupported cycle: False; unknown atom: False.')

Supported cycle: True; unsupported cycle: False; unknown atom: False.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")

solve_full_grid_fc: 0.29s
solve_full_grid_bc: 0.86s


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Answer:**

Let $I_{r,c,v}$ mean that cell $(r,c)$ contains value $v$. In the definite representation, let $N_{r,c,v}$ mean that value $v$ has been explicitly excluded from cell $(r,c)$.

### (a) General KB Strategy

The general representation uses `PropKB`. Only `Is` atoms are required because ordinary propositional negation can be applied directly to them.

For every cell $(r,c)$, the requirement that it contains at least one value is represented by:

$$
I_{r,c,1}\lor I_{r,c,2}\lor\cdots\lor I_{r,c,n}.
$$

For every cell and every pair of distinct values $v<w$, the at-most-one constraint is represented by:

$$
\neg I_{r,c,v}\lor\neg I_{r,c,w}.
$$

This clause prevents two different values from being assigned to the same cell simultaneously.

For every row $r$, value $v$, and pair of distinct columns $c_1<c_2$, row uniqueness is encoded as:

$$
\neg I_{r,c_1,v}\lor\neg I_{r,c_2,v}.
$$

For every column $c$, value $v$, and pair of distinct rows $r_1<r_2$, column uniqueness is encoded as:

$$
\neg I_{r_1,c,v}\lor\neg I_{r_2,c,v}.
$$

For every box, value $v$, and pair of distinct cells $(r_1,c_1)$ and $(r_2,c_2)$ in that box, box uniqueness is encoded as:

$$
\neg I_{r_1,c_1,v}\lor\neg I_{r_2,c_2,v}.
$$

Each given cell is added as a positive unit clause:

$$
I_{r,c,v}.
$$

All these expressions are already clauses in conjunctive normal form, so they can be inserted directly with `PropKB.tell()`.

For the default puzzle, the stored general KB contains:

- 81 cell-existence clauses;
- 2,916 within-cell uniqueness clauses;
- 2,916 row uniqueness clauses;
- 2,916 column uniqueness clauses;
- 2,916 box uniqueness clauses;
- 30 given facts.

This gives 11,775 stored clauses. Only 10,317 are distinct because two cells in the same box may also share a row or column. In those cases, the box rule repeats a clause already produced by the row or column rule. The duplicated clauses do not change the logical models of the KB, but they increase its storage and inference cost.

### (b) Definite KB Strategy

A definite clause contains exactly one positive conclusion. Therefore, the general cell-existence clause

$$
I_{r,c,1}\lor I_{r,c,2}\lor\cdots\lor I_{r,c,n}
$$

cannot be inserted into `PropDefiniteKB`, because it contains several positive literals.

The definite representation introduces explicit positive `Not` atoms. For example, `Not1_2_3` is a proposition meaning that value 3 has been excluded from cell $(1,2)$. It is an independent positive proposition, not classical negation applied to `Is1_2_3`.

If a cell is known to contain $v$, every other value is excluded from the same cell:

$$
I_{r,c,v}\Rightarrow N_{r,c,w},
\qquad w\ne v.
$$

If a cell contains $v$, that value is excluded from every other cell in the same row:

$$
I_{r,c,v}\Rightarrow N_{r,c',v},
\qquad c'\ne c.
$$

Column exclusion is encoded as:

$$
I_{r,c,v}\Rightarrow N_{r',c,v},
\qquad r'\ne r.
$$

The box rules only need to consider cells that share a box but share neither a row nor a column:

$$
I_{r,c,v}\Rightarrow N_{r',c',v},
\qquad r'\ne r,\quad c'\ne c.
$$

Same-row and same-column cells inside the box are already covered by the row and column rules. Excluding them from Rule 5 removes duplicate rules without weakening the KB.

The cell-existence requirement is handled through last-candidate reasoning. If all values except $v$ have been explicitly excluded from a cell, the remaining value is inferred:

$$
\left(
\bigwedge_{w\ne v}N_{r,c,w}
\right)
\Rightarrow I_{r,c,v}.
$$

Each given is inserted directly as an `Is` fact.

For the default puzzle, the definite KB contains:

- 729 last-candidate rules;
- 5,832 same-cell exclusion rules;
- 5,832 row exclusion rules;
- 5,832 column exclusion rules;
- 2,916 non-redundant box exclusion rules;
- 30 given facts.

The total is 21,171 facts and rules. All 21,171 are distinct, and every clause satisfies the definite-clause restriction.

Failure to prove an `Is` proposition does not automatically establish its corresponding `Not` proposition. Every explicit exclusion must itself be supported by a rule.


### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Answer:**

Neither the supplied truth-table implementation nor the supplied resolution implementation is suitable as the default full-grid Sudoku solver.

### Truth-Table Entailment

The general KB contains:

$$
9\times9\times9=729
$$

propositional `Is` symbols. Truth-table entailment may therefore need to examine as many as:

$$
2^{729}\approx 2.82\times10^{219}
$$

complete truth assignments.

This exponential model space is far beyond practical enumeration. The supplied implementation also does not use the propagation, clause learning and early pruning techniques used by modern SAT solvers.

### Resolution

The general KB contains 11,775 stored clauses. Resolution adds the negation of the query, producing 11,776 initial clauses.

An initial all-pairs pass can involve:

$$
\binom{11776}{2}
=
\frac{11776\times11775}{2}
=
69,331,200
$$

clause pairs.

New resolvents may enlarge the clause set and cause additional resolution rounds. Consequently, both execution time and memory consumption can grow rapidly.

### Experimental Interpretation

Construction of the general KB took approximately 0.157 seconds. However, neither `pl_resolution` nor `tt_entails` answered the directly given query `Is1_2_3` within the 30-second limit.

The timeout does not contradict soundness or completeness. These properties guarantee that an unrestricted completed execution returns the correct logical answer. They do not guarantee completion within a practical time or memory limit.

For these Sudoku puzzles, definite-clause reasoning is substantially more practical. The forward solver computes the complete definite closure once, while the backward solver performs goal-directed proof search.


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Answer:**

The backward-chaining algorithm treats the KB as an AND/OR proof graph.

- A goal succeeds if it is a known fact or if any rule concluding that goal succeeds.
- A rule succeeds only if all its premises succeed.

### Pseudocode

```text
BC-ENTAILS(KB, query):

    if the KB has not been indexed:
        facts = empty set
        rules_by_conclusion = empty mapping

        for each definite clause in KB:
            premises, conclusion = parse the clause

            if premises is empty:
                add conclusion to facts
            else:
                append premises to rules_by_conclusion[conclusion]

        proved = copy of facts

    repeat:
        old_proved_count = size(proved)
        failed_this_round = empty set
        active_path = empty set

        if PROVE(query):
            return True

        if size(proved) equals old_proved_count:
            return False


PROVE(goal):

    if goal is in proved:
        return True

    if goal is in failed_this_round:
        return False

    if goal is in active_path:
        return False

    add goal to active_path

    for each premise_list in rules_by_conclusion[goal]:

        rule_proved = True

        for each premise in premise_list:
            if PROVE(premise) is False:
                rule_proved = False
                stop checking this rule

        if rule_proved:
            remove goal from active_path
            add goal to proved
            return True

    remove goal from active_path
    add goal to failed_this_round
    return False
```

### Facts and Rule Selection

Known facts form the base case of recursion. When the current goal is already present in `proved`, the procedure returns `True` immediately.

Rules are indexed by their conclusion. Therefore, proving a goal only requires examining rules whose conclusion matches that goal.

### AND and OR Combination

The premises within one rule are combined using AND. Every premise must be proved before the conclusion can be accepted.

Multiple rules with the same conclusion are combined using OR. Proving any one matching rule is sufficient to prove the goal.

### Cycle Handling

The `active_path` set prevents a cycle from proving itself. Consider:

$$
A\Rightarrow B,
\qquad
B\Rightarrow A.
$$

Without an independent fact supporting either proposition, neither $A$ nor $B$ should be proved. When a goal already appears on the active recursive path, the current branch returns `False`.

The implemented tests produced:

- supported cycle: `True`;
- unsupported cycle: `False`;
- unknown proposition: `False`.

### Temporary Failures

A goal may initially fail but become provable after another branch establishes a new proposition. Consequently, unsuccessful goals are stored only in `failed_this_round`.

If a proof attempt adds new propositions to `proved`, the original query is attempted again with a new temporary failure set. Positive proofs remain valid and are retained.

### Termination

Termination follows from two finite restrictions:

1. `active_path` prevents infinite recursion within one proof branch.
2. A new outer iteration occurs only when the finite `proved` set has grown.

If a complete iteration fails to prove the query and produces no new proved propositions, the procedure returns `False`.

Facts, rule indexes and successful proofs are cached on the fixed KB instance and reused across later queries. This is valid for the current solver because the KB is not mutated after construction. A mutable KB would require cache invalidation after `tell()` or `retract()`.

For each of the five puzzles, all 729 possible `Is` queries agreed with the forward closure and the reference solution.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Answer:**

### Encoding Naked Pairs

Consider a Sudoku unit $U$, which may be a row, column or box. Let $x$ and $y$ be two distinct cells in $U$, and let $a$ and $b$ be two distinct values.

A naked pair exists when both cells have every value other than $a$ and $b$ explicitly excluded.

Define the premise:

$$
E_{x,y,a,b}
=
\bigwedge_{v\notin\{a,b\}}
\left(
N_{x,v}\land N_{y,v}
\right).
$$

For every other cell $z\in U\setminus\{x,y\}$, generate two definite rules:

$$
E_{x,y,a,b}\Rightarrow N_{z,a},
$$

$$
E_{x,y,a,b}\Rightarrow N_{z,b}.
$$

Every premise and conclusion is a positive `Not` proposition, so these rules satisfy the definite-clause restriction.

For example, suppose two cells in row 1 have every value except 3 and 7 excluded. Those cells must contain 3 and 7 in some order. Therefore, every other cell in row 1 can exclude both values.

### Consequences

Adding Naked Pair rules could derive exclusions unavailable to basic last-candidate reasoning. It could therefore solve puzzles that the basic elimination rules cannot complete.

However, exhaustive rule generation would substantially increase the KB size. A direct enumeration over all units, cell pairs, value pairs and target cells produces approximately:

$$
3n
\binom{n}{2}
\binom{n}{2}
2(n-2)
=
O(n^6)
$$

rules.

Each rule contains $2(n-2)$ premises, so the total number of stored premise occurrences can grow as:

$$
O(n^7).
$$

For $n=9$, naive generation produces 489,888 rules before removing overlaps and duplicate deductions.

Therefore, Naked Pairs are expressible using definite clauses, but targeted or on-demand generation would be more practical than adding every possible pattern in advance.



### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Answer:**

### (a) Scenario Where Backward Chaining Is Faster

Backward chaining is advantageous when the complete KB is large but the query depends on only a small subset of its rules.

For example, suppose a KB contains thousands of unrelated rule components, while the requested proposition depends on one fact and a short implication chain. With rules indexed by conclusion, backward chaining follows only rules that could derive the requested goal. Unrelated facts and rules are not processed.

A query for a given Sudoku cell is a simple example. The required proposition is already present in the fact set, so backward chaining can return immediately after the KB has been indexed.

This advantage is strongest when:

- only a small number of queries are required;
- each query has a small relevant dependency subgraph;
- rule conclusions are indexed;
- irrelevant parts of the KB are large.

### (b) Scenario Where Backward Chaining Has No Advantage

Backward chaining loses its advantage when a query depends on most of the KB, when a false query requires exploring many alternative rules, or when many related queries collectively require almost the complete deductive closure.

A complete Sudoku solution requires answers for many cells and candidate values. In this workload, computing one shared forward closure allows all derived facts to be reused for every cell.

The measured results were below：

In [11]:
results = []

for i,item in enumerate(puzzle_pool,start=1):
    start = perf_counter()
    fc = solve_full_grid_fc(n,box_h,box_w,item["givens"])
    fc_time = perf_counter() - start

    start = perf_counter()
    bc = solve_full_grid_bc(n,box_h,box_w,item["givens"])
    bc_time = perf_counter() - start

    assert fc == bc == item["solution"]

    results.append(
        (i,item["given_count"],fc_time,bc_time,bc_time / fc_time)
    )

table = [
    "| Puzzle | Givens | FC Time | BC Time | BC/FC |",
    "|---|---:|---:|---:|---:|"
]

for i,givens,fc_time,bc_time,ratio in results:
    table.append(
        f"| {i} | {givens} | {fc_time:.4f}s "
        f"| {bc_time:.4f}s | {ratio:.2f} |"
    )

display(Markdown("\n".join(table)))
total_fc = sum(row[2] for row in results)
total_bc = sum(row[3] for row in results)
ratio = total_bc / total_fc

print(f"Total FC time: {total_fc:.3f}s")
print(f"Total BC time: {total_bc:.3f}s")
print(f"Overall BC/FC ratio: {ratio:.2f}")

| Puzzle | Givens | FC Time | BC Time | BC/FC |
|---|---:|---:|---:|---:|
| 1 | 30 | 0.2629s | 0.5823s | 2.21 |
| 2 | 33 | 0.2521s | 0.5670s | 2.25 |
| 3 | 36 | 0.2538s | 0.5431s | 2.14 |
| 4 | 39 | 0.2723s | 0.5081s | 1.87 |
| 5 | 42 | 0.2550s | 0.4763s | 1.87 |

Total FC time: 1.296s
Total BC time: 2.677s
Overall BC/FC ratio: 2.07


Thus, backward chaining took approximately 2.43 times as long as forward chaining for this full-grid workload.

The forward solver is faster because forward_chain_closure performs one indexed propagation and reuses the complete closure for every cell. The backward solver starts from individual candidate goals and performs recursive rule exploration, although successful proofs are cached across queries.

This result is workload- and implementation-specific. It does not imply that forward chaining is always faster. Backward chaining may be preferable for one localized query, while a shared forward closure is more efficient when most consequences of the KB are required.

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

**Deployment pending:** the app has been verified locally. A cloud URL will be added after the intended GitHub account is selected, the private repository is created, and Streamlit deployment is verified. The local URL is not a deployed submission URL.


## References and implementation notes

The implementation uses the supplied course utilities unchanged. Reference materials were consulted for concepts and structure; the Sudoku builders, cycle-safe proof control, tests and app were written for this assignment.

- [AIMA Python](https://github.com/aimacode/aima-python): knowledge-base interfaces and the supplied-style inference algorithms.
- [Poole and Mackworth, AIPython](https://artint.info/AIPython/), sections 5.3–5.4: top-down AND/OR proofs and explaining supporting rules. The simple recursive procedure needs additional cycle handling for this assignment.
- [Peter Norvig, Solving Every Sudoku Puzzle](https://www.norvig.com/sudoku.html): units, peers and candidate elimination. Its search algorithm is not used here.
- [tyrelh/sudoku-as-sat](https://github.com/tyrelh/sudoku-as-sat): comparison with a CNF-to-SAT workflow; no external SAT solver is used here.
- [jhrcook/streamlit-sudoku](https://github.com/jhrcook/streamlit-sudoku): example of separating a Streamlit interface from its solver; its Pyomo engine is not used here.

The Streamlit application imports the shared implementation. Its loader retains only givens. `solution` is consulted exclusively by this notebook's validation. A `False` entailment answer means “not proved”; the app separately proves an explicit exclusion before presenting a negative explanation.
